# LLM: 次トークン予測から言語生成へ

LLMは、文脈から次のトークンを予測するDecoder-only Transformerである。文章全体を一度に作るのではなく、次トークン予測を繰り返す。


## このノートの読み方

想定読者: TransformerのSelf-Attentionと分類問題のCross Entropyを理解した学生。

MLPの次に読む教材として、直感、数式、shape、コード、HTMLアニメーション、`Trainer`学習例を往復しながら読む。


## MLPからの橋渡し

MLPの分類では1入力から1クラスを予測した。LLMでは各位置で語彙全体への分類を行う。実装では`labels=input_ids.clone()`として同じshapeで渡し、`forward`内で`logits[:, :-1]`と`labels[:, 1:]`を対応させる。


## 到達目標

- `input_ids`と`labels`のずれを説明できる
- causal maskの必要性を説明できる
- temperatureとtop-k samplingを説明できる


## 重要語句

- `logits`: softmax前の未正規化スコア
- `context window`: 一度に読めるトークン数
- `alignment`: モデル出力を人間の意図や安全性に近づける工程


## 準備

すべてのコードは小さなテンソルで概念を確認するためのものです。長い学習は行いません。


In [ ]:
from __future__ import annotations

import math

import numpy as np
import torch
from jaxtyping import Float
from torch import nn
from torch.utils.data import Dataset
import transformers
from transformers import Trainer, TrainingArguments

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)


## shape表

数式を読む前に、どのテンソルがどのshapeを持つかを固定する。

| 記号 | shape | 意味 |
|---|---|---|
| input_ids | (B, T) | 文脈トークン |
| labels | (B, T) | 1つ右へずれた正解 |
| logits | (B, T, vocab) | 各位置の語彙分類 |


## レビュー指摘を踏まえた補強

| 観点 | 補足 |
|---|---|
| 語彙分類 | 各位置では、語彙サイズをクラス数とする分類問題を解いている。`logits[:, :-1]`が予測、`labels[:, 1:]`が1つ右の正解である。 |
| Decoder-onlyの意味 | ここでは`TransformerEncoderLayer`にcausal maskを入れてDecoder-only相当の注意制約を作る。入門用の代用であり、実用LLMは専用decoder blockを積む。 |
| 位置情報 | 語順を扱うために位置埋め込みが必要である。位置なしでは同じbag-of-tokensに近づき、生成タスクとして弱くなる。 |
| 整列と評価 | 事前学習は次トークン予測、SFTは指示応答形式、選好最適化は人間の好みに寄せる段階である。 |
| バイオ用途 | 論文要約、実験ノート検索、タンパク質配列の次トークン予測などで同じ次トークン予測の形を使える。 |


## 次トークン予測

`Trainer`へ渡す`labels`は`input_ids`と同じshapeでよい。損失計算では予測側を最後以外、正解側を最初以外にずらして比較する。

$$
L=-\sum_t \log p(x_t\mid x_{<t})
$$


## Causal Mask

未来を見ない制約をAttention scoreに入れる。

$$
M_{ij}=0\ (j\le i),\quad M_{ij}=-\infty\ (j>i)
$$


## Sampling

temperatureは分布の鋭さを変え、top-kは候補を上位k個へ絞る。

$$
p_i=\frac{\exp(z_i/\tau)}{\sum_j\exp(z_j/\tau)}
$$


## 小さいテンソルで確認する

次のコードは、上の式がどのshapeを返すかを確認するための最小例である。


In [ ]:
text = "深層学習"
vocab = {ch: i for i, ch in enumerate(sorted(set(text + "は楽しい")))}
ids = torch.tensor([vocab[ch] for ch in text])
input_ids = ids[:-1]
labels = ids[1:]
print("vocab:", vocab)
print("input_ids:", input_ids.tolist())
print("labels:", labels.tolist())
logits = torch.randn(len(input_ids), len(vocab))
loss = nn.CrossEntropyLoss()(logits, labels)
print("next-token loss:", float(loss.detach()))


## 難所HTMLスライド

数式だけでは混ざりやすい箇所を、スライド形式で確認する。各スライドでは入力shape、計算、lossまたは生成手順への接続を1つずつ見る。


<p><a href="../demos/llm_difficulty_slides.html?v=20260522" target="_blank" rel="noopener">別タブで難所スライドを開く</a>（リポジトリ内: <code>demos/llm_difficulty_slides.html</code>）</p>
<iframe
  src="../demos/llm_difficulty_slides.html?v=20260522"
  width="100%"
  height="720"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="LLM: 次トークン予測から言語生成へ difficulty slides"
></iframe>


## HTMLアニメーションで確認する

以下のHTMLは`teaching-html-animation` skillの方針に合わせ、各状態を式・shape・コード上の概念に結びつけている。


### tokenization and shift animation

- 学習目標: 文章からinput_idsとlabelsを作る
- 誤解の防止: labels=input_idsでよいと誤解する

対応する式:

$$
\text{labels}_{t}=\text{input}_{t+1}
$$


<p><a href="../demos/llm_token_shift.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/llm_token_shift.html</code>）</p>
<iframe
  src="../demos/llm_token_shift.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="tokenization and shift animation"
></iframe>


### causal mask animation

- 学習目標: 未来トークンをAttentionから隠す
- 誤解の防止: 学習時に未来を見てもよいと思う

対応する式:

$$
S'=S+M
$$


<p><a href="../demos/llm_causal_mask.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/llm_causal_mask.html</code>）</p>
<iframe
  src="../demos/llm_causal_mask.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="causal mask animation"
></iframe>


### logits softmax sampling animation

- 学習目標: logitsから確率とサンプルを作る
- 誤解の防止: logitsを確率と誤解する

対応する式:

$$
p=\mathrm{softmax}(z/\tau)
$$


<p><a href="../demos/llm_sampling.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/llm_sampling.html</code>）</p>
<iframe
  src="../demos/llm_sampling.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="logits softmax sampling animation"
></iframe>


### autoregressive generation animation

- 学習目標: 予測tokenを文脈へ戻す反復を見せる
- 誤解の防止: 文章を一括生成すると思う

対応する式:

$$
x_{t+1}\sim p(\cdot\mid x_{\le t})
$$


<p><a href="../demos/llm_autoregressive.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/llm_autoregressive.html</code>）</p>
<iframe
  src="../demos/llm_autoregressive.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="autoregressive generation animation"
></iframe>


## `Trainer`で学習する

この章の`Trainer`例は、汎用MSE回帰ではなく、`LLM: 次トークン予測から言語生成へ`固有のデータ形式とlossを返す。基本は標準の`Trainer(model, args, train_dataset)`を使い、`forward`が`loss`と`logits`を返す形にそろえる。GANはD/Gでoptimizerを分ける必要があるため`Trainer`を継承した交互更新デモ、DBMは平均場CDサロゲートとして扱う。


In [ ]:
class TinyCausalLMDataset(Dataset):
    def __init__(self, n_samples: int = 40, seq_len: int = 8, vocab_size: int = 15) -> None:
        starts = torch.randint(0, vocab_size, (n_samples, 1))
        offsets = torch.arange(seq_len).view(1, -1)
        self.input_ids = (starts + offsets) % vocab_size
        self.labels = self.input_ids.clone()

    def __len__(self) -> int:
        return len(self.input_ids)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        return {"input_ids": self.input_ids[index], "labels": self.labels[index]}


class TinyDecoderOnlyLM(nn.Module):
    def __init__(self, vocab_size: int = 15, d_model: int = 16, max_len: int = 8) -> None:
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.block = nn.TransformerEncoderLayer(d_model=d_model, nhead=2, dim_feedforward=32, batch_first=True)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids: torch.Tensor, labels: torch.Tensor | None = None) -> dict[str, torch.Tensor]:
        positions = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)
        hidden = self.token_embedding(input_ids) + self.position_embedding(positions)
        seq_len = input_ids.shape[1]
        causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=input_ids.device), diagonal=1).bool()
        hidden = self.block(hidden, src_mask=causal_mask)
        logits = self.lm_head(hidden)
        loss = None
        if labels is not None:
            pred = logits[:, :-1].reshape(-1, logits.size(-1))
            target = labels[:, 1:].reshape(-1)
            loss = nn.CrossEntropyLoss()(pred, target)
        return {"loss": loss, "logits": logits}


training_args = TrainingArguments(
    output_dir="./results/llm_trainer_demo",
    max_steps=3,
    per_device_train_batch_size=8,
    learning_rate=1e-3,
    logging_strategy="no",
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
    seed=SEED,
    use_cpu=not torch.cuda.is_available(),
)

dataset = TinyCausalLMDataset()
trainer = Trainer(model=TinyDecoderOnlyLM(), args=training_args, train_dataset=dataset)
train_output = trainer.train()
context = torch.tensor([[1, 2, 3]])
for _ in range(3):
    with torch.no_grad():
        logits = trainer.model(context)["logits"][:, -1]
        next_id = torch.argmax(logits, dim=-1, keepdim=True)
    context = torch.cat([context, next_id], dim=1)
print("Causal LM Trainer loss:", train_output.training_loss)
print("greedy generated ids:", context.tolist())


## 系列モデルとしての位置づけ

| モデル | 学習目的 | mask | 主な出力 | つまずき |
|---|---|---|---|---|
| MLP | 固定長ベクトルの分類・回帰 | なし | class/logit | 系列順序を扱いにくい |
| Transformer Encoder | 系列全体の表現学習 | padding maskなど | 文脈表現 | `T x T` attentionのshape |
| Decoder-only LLM | 次トークン予測 | causal mask | `B x T x vocab` logits | label shiftとsampling |


## 発展課題

- 小語彙で生成ループを書く
- perplexityを計算する
- SFTとpretrainingのデータ形式を比べる


## 確認問題

- `input_ids=[2,5,7,1]`の次トークンラベルを書く。
- temperatureを下げると分布はどう変わるか。


## まとめ

- MLPから何が変わったのかを、shapeとlossで確認する。
- HTMLアニメーションは式の代わりではなく、式とコードを読むための補助である。
- `Trainer`は学習ループを隠すが、`Dataset`のキー、`forward`の引数、`loss`の意味は必ず確認する。
